In [1]:
# ===========================================
# ONE-BLOCK LCFT MASTER RUN (Self-Contained)
# ===========================================

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
from google.colab import drive
%matplotlib inline

# -------------------------------
# 1. MOUNT DRIVE + PROJECT FOLDER
# -------------------------------
drive.mount('/content/drive')
PROJECT = "/content/drive/MyDrive/LCFT_Project"
os.makedirs(PROJECT, exist_ok=True)
print(f"\nProject folder ready: {PROJECT}\n")

# -------------------------------
# 2. LCFT CONSTANTS
# -------------------------------
DELTA_STAR = 0.147520
K_BETA     = 0.065292
FLOW_TIME  = 130.0
NOISE      = 5e-5
USE_BETA_FLOW = False

np.random.seed(12345)

print("LCFT URT Master Code Loaded (Discrete Mode)")
print(f"δ* = {DELTA_STAR} | kβ = {K_BETA} | t_flow = {FLOW_TIME}")
print("Mode: Discrete URT\n")

# -------------------------------
# 3. CORE FUNCTIONS
# -------------------------------
def lcft_delta(x):
    x = np.asarray(x).ravel()
    dx = np.abs(np.diff(x))
    return np.mean(dx) / (np.std(x) + 1e-8)

def urt_stabilize_1d(x, delta_target=0.1, alpha=0.8, beta=0.8,
                     theta_H=0.2, k=0.5, window=50):
    x = np.asarray(x).ravel()
    y = x.copy()
    denom = 1.0 - alpha*beta*(1 + theta_H)
    if denom <= 0:
        raise ValueError("Bad URT config")

    for t in range(2, len(y)):
        phi = y[t-1] - y[t-2]
        seg = y[max(0, t-window):t]
        seg_std = np.std(seg) + 1e-8
        delta_t = abs(phi) / (seg_std * denom + 1e-8)
        u = k*(delta_target - delta_t)
        y[t] = beta*(alpha*(y[t-1] - theta_H*phi) + u)
    return y

def beta_flow_collapse(raw, delta_star=DELTA_STAR,
                       k_beta=K_BETA, t=FLOW_TIME):
    return delta_star + (raw - delta_star)*np.exp(-k_beta*t) + np.random.normal(0, NOISE)

# -------------------------------
# 4. CHAOTIC GENERATORS
# -------------------------------
def logistic_map(N=5000, r=3.9):
    x = np.zeros(N); x[0] = 0.2
    for i in range(N-1):
        x[i+1] = r*x[i]*(1-x[i])
    return x

def sine_map(N=5000, a=0.9):
    x = np.zeros(N); x[0] = 0.2
    for i in range(N-1):
        x[i+1] = a*np.sin(np.pi*x[i])
    return x

def rnn_1d(N=5000, w=1.2):
    x = np.zeros(N); x[0] = 0.2
    for i in range(N-1):
        x[i+1] = np.tanh(w*x[i]) + 0.03*np.random.randn()
    return x

def make_system(i):
    if i % 3 == 0:
        return f"LOGISTIC_{i:03d}", "logistic", logistic_map
    elif i % 3 == 1:
        return f"SINE_{i:03d}", "sine", sine_map
    else:
        return f"RNN_{i:03d}", "rnn", rnn_1d

# -------------------------------
# 5. SCAN RUNNER
# -------------------------------
def run_scan(n, name):
    print(f"\nRunning scan: {n} systems...")
    rows = []

    for i in tqdm(range(1, n+1)):
        sys_name, kind, gen = make_system(i)
        x = gen()
        raw = lcft_delta(x)

        if USE_BETA_FLOW:
            urt = beta_flow_collapse(raw)
        else:
            y = urt_stabilize_1d(x)
            urt = lcft_delta(y)

        rows.append({
            "index": i,
            "name": sys_name,
            "kind": kind,
            "raw_delta": raw,
            "urt_delta": urt,
            "collapse_error": abs(urt - DELTA_STAR)
        })

    df = pd.DataFrame(rows)

    # Save results
    out_csv = f"{PROJECT}/{name}_scan.csv"
    df.to_csv(out_csv, index=False)
    print(f"Saved CSV → {out_csv}")

    # Histogram
    plt.figure(figsize=(8,5))
    plt.hist(df["raw_delta"], bins=50, alpha=0.5, label="Raw")
    plt.hist(df["urt_delta"], bins=50, alpha=0.5, label="URT")
    plt.axvline(DELTA_STAR, color="red", linestyle="--")
    plt.legend()
    plt.title(f"{name} Distribution")
    out_png = f"{PROJECT}/{name}_hist.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved Histogram → {out_png}")

    return df

# -------------------------------
# 6. EXECUTE BOTH SCANS
# -------------------------------
df1000 = run_scan(1000, "lcft_1000")
df5000 = run_scan(5000, "lcft_5000")

print("\n===== MASTER SUMMARY (5000 SYSTEMS) =====")
print(df5000["urt_delta"].describe())
print("\nComplete. All files saved.")

Mounted at /content/drive

Project folder ready: /content/drive/MyDrive/LCFT_Project

LCFT URT Master Code Loaded (Discrete Mode)
δ* = 0.14752 | kβ = 0.065292 | t_flow = 130.0
Mode: Discrete URT


Running scan: 1000 systems...


100%|██████████| 1000/1000 [02:38<00:00,  6.33it/s]


Saved CSV → /content/drive/MyDrive/LCFT_Project/lcft_1000_scan.csv
Saved Histogram → /content/drive/MyDrive/LCFT_Project/lcft_1000_hist.png

Running scan: 5000 systems...


100%|██████████| 5000/5000 [12:51<00:00,  6.48it/s]


Saved CSV → /content/drive/MyDrive/LCFT_Project/lcft_5000_scan.csv
Saved Histogram → /content/drive/MyDrive/LCFT_Project/lcft_5000_hist.png

===== MASTER SUMMARY (5000 SYSTEMS) =====
count    5000.000000
mean        0.146899
std         0.000170
min         0.146017
25%         0.146851
50%         0.146851
75%         0.147047
max         0.147529
Name: urt_delta, dtype: float64

Complete. All files saved.
